In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import TimestampType, IntegerType, DateType, FloatType, DecimalType
from pyspark.sql.functions import col, initcap, trim, when


def standarize_market_prices(df: DataFrame) -> DataFrame:
    return ( df
            .withColumn("event_date", col('event_date').cast(DateType()))
            .withColumn("last_updated_ts", col('last_updated_ts').cast(TimestampType()))
            .withColumn('market_type', trim(col('market_type')))
            .withColumn('price_eur_mwh', col('price_eur_mwh').cast(DecimalType(10, 2)))
            .withColumn('volume_mwh', col('volume_mwh').cast(DecimalType(10, 2)))
            .drop(col("_rescued_data"))
    )

def get_revenue_eur(df: DataFrame) -> DataFrame:
    return (df.withColumn("revenue_eur", col("price_eur_mwh") * col("volume_mwh")))


def get_price_category(df: DataFrame) -> DataFrame:
    quantiles = df.approxQuantile("price_eur_mwh", [0.33, 0.66], 0.01)
    p33, p66 = quantiles[0], quantiles[1]

    return (df.withColumn("price_category",
        when(col("price_eur_mwh") <= p33, "Low")
        .when(col("price_eur_mwh") <= p66, "Medium")
        .otherwise("High")
    ))


def get_volume_category(df: DataFrame) -> DataFrame:
    quantiles_vol = df.approxQuantile("volume_mwh", [0.33, 0.66], 0.01)
    v33, v66 = quantiles_vol[0], quantiles_vol[1]

    return (df.withColumn("volume_category",
        when(col("volume_mwh") <= v33, "Low_Volume")
        .when(col("volume_mwh") <= v66, "Medium_Volume")
        .otherwise("High_Volume")
    ))

def drop_meaningless_cols(df: DataFrame) -> DataFrame:
    return (
        df.
        drop(col("_rescued_data"))
    )